In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

Приведем все даты к нужному типу

In [13]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])
order_reviews["review_creation_date"] = pd.to_datetime(order_reviews["review_creation_date"])
order_reviews["review_answer_timestamp"] = pd.to_datetime(order_reviews["review_answer_timestamp"])
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

Далее переходим к созданию признаков для полного анализа:

Месяц и год:

In [15]:
orders["order_month"] = orders["order_purchase_timestamp"].dt.month
orders["order_year"] = orders["order_purchase_timestamp"].dt.year
orders[["order_purchase_timestamp", "order_month", "order_year"]].head()

,order_purchase_timestamp,order_month,order_year
0,2017-10-02 10:56:33,10,2017
1,2018-07-24 20:41:37,7,2018
2,2018-08-08 08:38:49,8,2018
3,2017-11-18 19:28:06,11,2017
4,2018-02-13 21:18:39,2,2018


Факт. срок доставки, ожидаемый срок доставки, задержка доставки и есть ли задержка:

In [17]:
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
orders["expected_delivery_days"] = (orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]).dt.days
orders["delay_days"] = orders["delivery_days"] - orders["expected_delivery_days"]
orders["is_late"] = orders["delay_days"] > 0

Создадим функцию которая будет выявлять категорию задержки:

In [18]:
def delay_group(days):
    if pd.isna(days):
        return "unknown"
    elif days <= 3:
        return "0-3 days"
    elif days <= 7:
        return "4-7 days"
    else:
        return ">7 days"

In [19]:
orders["delay_group"] = orders["delay_days"].apply(delay_group)
orders["expected_group"] = orders["expected_delivery_days"].apply(delay_group)

Флаги позитивного, нейтрального и негативного отзыва:

In [21]:
order_reviews["positive_review"] = order_reviews["review_score"] >= 4
order_reviews["neutral_review"] = order_reviews["review_score"] == 3
order_reviews["negative_review"] = order_reviews["review_score"] <= 2

Создадим признак общей стоимости заказа (товар + доставка):

In [23]:
order_items["item_total"] = order_items["price"] + order_items["freight_value"]

Обьединим таблицы все в одну общую и оставим только последний отзыв от клиента(если их несколько):

In [33]:
order_reviews_latest = (order_reviews.sort_values("review_answer_timestamp").drop_duplicates(subset="order_id", keep="last"))


In [37]:
order_items_mart = (
    order_items
    .merge(orders, on="order_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(products, on="product_id", how="left")
    .merge(category_translation, on="product_category_name", how="left")
    .merge(sellers, on="seller_id", how="left")
    .merge(order_reviews_latest[["order_id", "review_score", "positive_review", "neutral_review", "negative_review"]], on="order_id", how="left")
)

In [39]:
print("Строк в order_items:", order_items.shape[0])
print("Строк в order_items_mart:", order_items_mart.shape[0])

Строк в order_items: 112650
Строк в order_items_mart: 112650


В таблице заказой один заказ может иметь несколько платёжных записей. Чтобы не размножить строки основной витрины, сначала агрегируем оплаты до уровня одного заказа:.

In [40]:
payments_by_order = (order_payments.groupby("order_id", as_index=False).agg(
        payment_total=("payment_value", "sum"),
        payment_records=("payment_sequential", "count"),
        max_installments=("payment_installments", "max"),
        main_payment_type=("payment_type", lambda x: x.mode().iloc[0])
    )
)

Теперь присоединим оплаты к общей витрине:

In [43]:
order_items_mart = order_items_mart.merge(payments_by_order, on="order_id", how="left")

В данных есть товары без категории или без английского перевода категории, их заполним значениями: "Unknown"

In [47]:
order_items_mart["product_category_name"] = order_items_mart["product_category_name"].fillna("Unknown")
order_items_mart["product_category_name_english"] = order_items_mart["product_category_name_english"].fillna("Unknown")

Все данные вставили корректно и без дублей, количество строк осталось прежнем, можно сохранять эти данные в data/processed.

In [52]:
processed_path = Path("../data/processed")
order_items_mart.to_csv(processed_path / "order_items_mart.csv", index=False)
customers_mart = (order_items_mart.groupby("customer_id", as_index=False).agg(
        total_orders=("order_id", "nunique"),
        total_items=("product_id", "count"),
        total_spent=("item_total", "sum"),
        avg_order_value=("item_total", "mean"),
        positive_reviews=("positive_review", "sum"),
        negative_reviews=("negative_review", "sum")
    )
)
customers_mart.to_csv(processed_path / "customers_mart.csv", index=False)
sellers_mart = (order_items_mart.groupby("seller_id", as_index=False).agg(
        total_items_sold=("product_id", "count"),
        total_revenue=("item_total", "sum"),
        positive_reviews=("positive_review", "sum"),
        negative_reviews=("negative_review", "sum")
    )
)
sellers_mart.to_csv(processed_path / "sellers_mart.csv", index=False)
categories_mart = (order_items_mart.groupby("product_category_name_english", as_index=False).agg(
        total_items_sold=("product_id", "count"),
        total_revenue=("item_total", "sum"),
        positive_reviews=("positive_review", "sum"),
        negative_reviews=("negative_review", "sum")
    )
)
categories_mart.to_csv(processed_path / "categories_mart.csv", index=False)
dashboard_mart = (order_items_mart.groupby(["order_year", "order_month"], as_index=False).agg(
        total_orders=("order_id", "nunique"),
        total_items=("product_id", "count"),
        total_revenue=("item_total", "sum"),
        avg_order_value=("item_total", "mean"),
        positive_reviews=("positive_review", "sum"),
        negative_reviews=("negative_review", "sum")
    )
)
dashboard_mart.to_csv(processed_path / "dashboard_mart.csv", index=False)